# OMNI-905 · The PFM trained on CCD and UCF-Crime together; the epoch and the thresholds on validation (Colab GPU)

Runs the repo's own code from a clone of the branch: `scripts/pfm_model.py` (Algorithm 4, D-56's config),
`scripts/train_pfm.py` (training, resume, the measures), `scripts/pfm_samples.py` (the committed samples) and
`scripts/pfm_runs.py` (the run record). The rules were written before training (D-59):

- **One model, both domains:** the shared backbone and a head per domain; each step one batch from each domain's
  train split, losses summed (B.4). D-56's config, 30 epochs, **3 seeds**; one change: each domain's `Δt` term is
  weighed by `1 / var(τ)` of its train positives (UCF-Crime's `τ` reaches 67 s, CCD's 4.6 s)
- **The choice, on validation only:** after every epoch, the ROC-AUC of "an event" against Normal on UCF-Crime's
  validation samples; the best (seed, epoch) of the 90 is the model. CCD is measured too, but it is at its ceiling
  (K-35), so it cannot choose
- **The thresholds `θ_d`:** on the chosen model, per domain, the lowest `θ` at which at most 10% of the domain's
  validation negative windows warn

1. **Setup:** clone, Drive
2. **The vectors:** both domains' InternVideo2 folders copied from Drive to this machine, and the samples
3. **Training:** 3 seeds; checkpoints on Drive (`models/pfm/joint/seed*/`): if the session ends, Run all again and
   every run resumes
4. **The choice and the record:** the chosen epoch's weights → `models/pfm/runs/{run_id}/model.pt`, its `run.json`
   (config, epochs, thresholds, validation) and `models/pfm/model_runs.json`; `results/omni905_joint.json`

Runtime → Change runtime type → **GPU**. Held-out data is not touched (OMNI-906).

In [ ]:
# 1 · Setup
BRANCH = "slice-09-10-forecast"
REPO = "https://github.com/OmniSight-team/omnisight.git"

import os, subprocess, sys
if not os.path.isdir("/content/omni-sight"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/omni-sight"], check=True)
else:
    subprocess.run(["git", "-C", "/content/omni-sight", "pull", "-q"], check=True)
COMMIT = subprocess.run(["git", "-C", "/content/omni-sight", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)
sys.path.insert(0, "/content/omni-sight/scripts")

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU"
DEVICE, GPU = "cuda", torch.cuda.get_device_name(0)
print(torch.__version__, GPU)

In [ ]:
# 2 · Both domains' InternVideo2 vectors, copied from Drive once, and the samples
import hashlib, json, shutil, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pfm_runs, pfm_samples, train_pfm
from pfm_model import Config

REPO_DIR = Path("/content/omni-sight")
DRIVE = Path("/content/drive/MyDrive/omnisight")
LOCAL = Path("/content/pfm")
DOMAINS = ("ccd", "ucf_crime")
SAMPLE_FILES = {d: REPO_DIR / "datasets" / "samples" / f"{d}.csv" for d in DOMAINS}
SAMPLES = {d: pfm_samples.read_samples(SAMPLE_FILES[d]) for d in DOMAINS}

def copy_folder(domain):
    keys = sorted({s["clip_key"] for s in SAMPLES[domain]})
    source, target = DRIVE / "embeddings" / "pfm" / domain / "internvideo2", LOCAL / domain
    def one(key):
        dst = target / f"{key}.npz"
        if not dst.is_file():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source / f"{key}.npz", dst)
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(one, keys))
    return target

STORES, TRAIN, VAL = {}, {}, {}
for d in DOMAINS:
    start = time.time()
    STORES[d] = pfm_samples.Store(SAMPLES[d], copy_folder(d))
    TRAIN[d] = train_pfm.Data.split(STORES[d], SAMPLES[d], "train", d)
    VAL[d] = train_pfm.Data.split(STORES[d], SAMPLES[d], "validation", d)
    print(f"{d}: {len(STORES[d].vectors):,} vectors, train {len(TRAIN[d]):,} samples, validation {len(VAL[d]):,} "
          f"({time.time() - start:.0f} s)")

In [ ]:
# 3 · Training: one model for both domains, 3 seeds, validation after every epoch
SEEDS = (0, 1, 2)
# D-59: each domain's Δt term weighed by 1 / var(τ) on its train positives, so "always the mean τ" costs 1 in both
# domains (UCF-Crime's τ reaches 67 s; D-56's α = 1 was set on CCD's, under 5 s). Every other setting as D-56.
ALPHAS = {d: float(f"{1.0 / np.nanvar(TRAIN[d].tau):.3g}") for d in DOMAINS}
CONFIG = Config(input_dim=512, domains=list(DOMAINS), alphas=ALPHAS)
print("alphas", ALPHAS)
CHOOSE_BY = "ucf_crime"  # D-59: CCD is at its ceiling (K-35)
RUNS_DIR = DRIVE / "models" / "pfm" / "joint"

def validate(model):
    out = {}
    for d in DOMAINS:
        p, dt = train_pfm.predict(model, VAL[d], DEVICE)
        out[d] = train_pfm.evaluate(VAL[d], p, dt)
    out["score"] = out[CHOOSE_BY]["roc_auc"]
    return out

HISTORY = {}
for seed in SEEDS:
    print(f"seed {seed}")
    _, HISTORY[seed] = train_pfm.train(CONFIG, dict(TRAIN), RUNS_DIR / f"seed{seed}", seed=seed, device=DEVICE,
                                       validate=validate)

In [ ]:
# 4 · The choice (the best seed and epoch on validation), the thresholds, the record
best = {seed: max(HISTORY[seed], key=lambda h: (h["validation"]["score"], -h["epoch"])) for seed in SEEDS}
for seed, h in best.items():
    v = h["validation"]
    print(f"seed {seed}: best epoch {h['epoch']}, UCF-Crime AUC {v['ucf_crime']['roc_auc']:.4f}, "
          f"CCD AUC {v['ccd']['roc_auc']:.4f}; last epoch {HISTORY[seed][-1]['validation']['score']:.4f}")
SEED = max(SEEDS, key=lambda s: (best[s]["validation"]["score"], -s))
model, state = train_pfm.load(RUNS_DIR / f"seed{SEED}" / "best.pt", DEVICE)
assert state["epoch"] == best[SEED]["epoch"]

VALIDATION = {}
for d in DOMAINS:
    p, dt = train_pfm.predict(model, VAL[d], DEVICE)
    VALIDATION[d] = m = train_pfm.evaluate(VAL[d], p, dt)
    print(f"{d:9s} AUC {m['roc_auc']:.4f}  top-1 {m['top1']:.4f}  top-3 (positives) {m['top3_positive']:.4f}  "
          f"Δt MAE {m['dt_mae_s']:.2f} s  θ {m['theta']:.4f}  false alarms {m['false_alarm_windows']:.3f} of "
          f"{m['negative_windows']} windows  warned {m['warned_windows']}/{m['positive_windows']}  lead {m['lead_time_s'] or 0:.2f} s")
assert abs(VALIDATION[CHOOSE_BY]["roc_auc"] - best[SEED]["validation"]["score"]) < 1e-6  # the reloaded model is the chosen one

RUN_ID = f"pfm-joint-v1-s{SEED}-e{state['epoch']}"
folder = DRIVE / pfm_runs.RUNS / RUN_ID
folder.mkdir(parents=True, exist_ok=True)
shutil.copyfile(RUNS_DIR / f"seed{SEED}" / "best.pt", folder / pfm_runs.CHECKPOINT)
run = pfm_runs.record(DRIVE, RUN_ID, config=CONFIG.to_dict(), epochs_trained=CONFIG.epochs, epoch=state["epoch"], seed=SEED,
                      thresholds={d: VALIDATION[d]["theta"] for d in DOMAINS},
                      validation={"score": best[SEED]["validation"]["score"], "chosen_by": f"{CHOOSE_BY} roc_auc", **VALIDATION},
                      commit=COMMIT, gpu=GPU, torch=torch.__version__, encoder="internvideo2",
                      false_alarm_rule=train_pfm.FALSE_ALARM,
                      samples_sha256={d: hashlib.sha256(SAMPLE_FILES[d].read_bytes()).hexdigest() for d in DOMAINS},
                      created_at=time.strftime("%Y-%m-%d %H:%M"))
pfm_runs.register(DRIVE, run, chosen=True)
print(RUN_ID, run["thresholds"])

out = DRIVE / "results" / "omni905_joint.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({"commit": COMMIT, "gpu": GPU, "torch": torch.__version__, "config": CONFIG.to_dict(),
    "seeds": SEEDS, "chosen": {"run_id": RUN_ID, "seed": SEED, "epoch": state["epoch"]},
    "best_per_seed": {s: {"epoch": h["epoch"], "score": h["validation"]["score"]} for s, h in best.items()},
    "history": HISTORY, "train_seconds": {s: sum(h["seconds"] for h in HISTORY[s]) for s in SEEDS},
    "validation": VALIDATION}, indent=1, default=float))
print(out)